In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('../data/ready_to_train.csv')

In [7]:
with pd.option_context('display.max_columns', None):
    display(df.head())

,loan_amnt,term,int_rate,sub_grade,emp_length,annual_inc,verification_status,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,target,mths_since_earliest_cr_line,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding
0,5000,36,10.65,6,10,24000.0,2,27.65,0.0,1.0,3.0,0.0,13648,83.7,9.0,0,0.0,0.0,0.0,81518.5,22900.0,0,497,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
1,2500,60,15.27,13,0,30000.0,1,1.00,0.0,5.0,3.0,0.0,1687,9.4,4.0,0,0.0,0.0,0.0,81518.5,22900.0,1,326,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2400,36,15.96,14,10,12252.0,0,8.72,0.0,2.0,2.0,0.0,2956,98.5,10.0,0,0.0,0.0,0.0,81518.5,22900.0,0,295,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0
3,10000,36,13.49,10,10,49200.0,1,20.00,0.0,1.0,10.0,0.0,5598,21.0,37.0,0,0.0,0.0,0.0,81518.5,22900.0,0,364,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
4,3000,60,12.69,9,1,80000.0,1,17.94,0.0,0.0,15.0,0.0,27783,53.9,38.0,0,0.0,0.0,0.0,81518.5,22900.0,0,365,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0


In [4]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== Hasil Split Data ===")
print(f"Total Data Fitur Train (X_train) : {X_train.shape}")
print(f"Total Data Label Train (y_train) : {y_train.shape}")
print(f"Total Data Fitur Test  (X_test)  : {X_test.shape}")
print(f"Total Data Label Test  (y_test)  : {y_test.shape}")

print("\n=== Cek Proporsi Target (Stratify) ===")
print(f"Proporsi di y_train:\n{y_train.value_counts(normalize=True)}")
print(f"Proporsi di y_test:\n{y_test.value_counts(normalize=True)}")

=== Hasil Split Data ===
Total Data Fitur Train (X_train) : (372665, 40)
Total Data Label Train (y_train) : (372665,)
Total Data Fitur Test  (X_test)  : (93167, 40)
Total Data Label Test  (y_test)  : (93167,)

=== Cek Proporsi Target (Stratify) ===
Proporsi di y_train:
target
0    0.8814
1    0.1186
Name: proportion, dtype: float64
Proporsi di y_test:
target
0    0.881396
1    0.118604
Name: proportion, dtype: float64


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [9]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("=== Proporsi y_train setelah SMOTE ===")
print(y_train_smote.value_counts())

=== Proporsi y_train setelah SMOTE ===
target
0    328467
1    328467
Name: count, dtype: int64


In [11]:
from sklearn.linear_model import LogisticRegression


model_lr = LogisticRegression(random_state=42, max_iter=1000)

model_lr.fit(X_train_smote, y_train_smote)

y_pred = model_lr.predict(X_test_scaled)

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

=== Confusion Matrix ===
[[50873 31244]
 [ 4023  7027]]

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.93      0.62      0.74     82117
           1       0.18      0.64      0.28     11050

    accuracy                           0.62     93167
   macro avg       0.56      0.63      0.51     93167
weighted avg       0.84      0.62      0.69     93167



In [16]:
from xgboost import XGBClassifier

# 1. Panggil model XGBoost
model_xgb = XGBClassifier(random_state=42)

# 2. Latih model menggunakan data latih yang sudah di-SMOTE
model_xgb.fit(X_train_smote, y_train_smote)

# 3. Minta model menebak data uji
y_pred_xgb = model_xgb.predict(X_test_scaled)

# 4. Cetak hasil evaluasi
print("=== Confusion Matrix XGBoost ===")
print(confusion_matrix(y_test, y_pred_xgb))

print("\n=== Classification Report XGBoost ===")
print(classification_report(y_test, y_pred_xgb))

=== Confusion Matrix XGBoost ===
[[81995   122]
 [10976    74]]

=== Classification Report XGBoost ===
              precision    recall  f1-score   support

           0       0.88      1.00      0.94     82117
           1       0.38      0.01      0.01     11050

    accuracy                           0.88     93167
   macro avg       0.63      0.50      0.47     93167
weighted avg       0.82      0.88      0.83     93167



In [18]:
rasio = float(y_train.value_counts()[0]) / y_train.value_counts()[1]
print(f"Nilai scale_pos_weight yang digunakan: {rasio:.2f}")

model_xgb_balanced = XGBClassifier(random_state=42, scale_pos_weight=rasio)

model_xgb_balanced.fit(X_train_scaled, y_train)

y_pred_xgb_bal = model_xgb_balanced.predict(X_test_scaled)

from sklearn.metrics import classification_report, confusion_matrix
print("\n=== Confusion Matrix XGBoost (Balanced) ===")
print(confusion_matrix(y_test, y_pred_xgb_bal))

print("\n=== Classification Report XGBoost (Balanced) ===")
print(classification_report(y_test, y_pred_xgb_bal))

Nilai scale_pos_weight yang digunakan: 7.43

=== Confusion Matrix XGBoost (Balanced) ===
[[52753 29364]
 [ 4092  6958]]

=== Classification Report XGBoost (Balanced) ===
              precision    recall  f1-score   support

           0       0.93      0.64      0.76     82117
           1       0.19      0.63      0.29     11050

    accuracy                           0.64     93167
   macro avg       0.56      0.64      0.53     93167
weighted avg       0.84      0.64      0.70     93167



In [20]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Tentukan rentang parameter yang mau diuji
param_grid = {
    'max_depth': [3, 5, 7],               # Kedalaman pohon
    'learning_rate': [0.01, 0.1, 0.2],    # Kecepatan belajar
    'n_estimators': [100, 200]            # Jumlah pohon
}

# 2. Panggil model dasar (tetap gunakan scale_pos_weight dari perhitungan sebelumnya!)
xgb_base = XGBClassifier(random_state=42, scale_pos_weight=rasio)

# 3. Siapkan alat pencarinya
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=5,              # Menguji 5 kombinasi acak
    scoring='recall',      # FOKUS mencari Recall terbaik
    cv=3,                  # Validasi silang 3 kali per kombinasi
    random_state=42,
    n_jobs=-1              # Gunakan seluruh core prosesor agar cepat
)

# 4. Mulai pencarian di data latih asli
print("Mulai mencari kombinasi terbaik (Tunggu sebentar ya)...")
random_search.fit(X_train_scaled, y_train)

# 5. Tampilkan hasil kombinasi terbaik
print("\n=== Setelan Parameter Terbaik ===")
print(random_search.best_params_)

Mulai mencari kombinasi terbaik (Tunggu sebentar ya)...

=== Setelan Parameter Terbaik ===
{'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.01}


In [21]:
model_xgb_tuned = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.01,
    scale_pos_weight=rasio,
    random_state=42
)

# 2. Latih model final menggunakan data latih asli
model_xgb_tuned.fit(X_train_scaled, y_train)

# 3. Minta model menebak data uji
y_pred_xgb_tuned = model_xgb_tuned.predict(X_test_scaled)

# 4. Cetak hasil evaluasi
print("\n=== Confusion Matrix XGBoost (Tuned) ===")
print(confusion_matrix(y_test, y_pred_xgb_tuned))

print("\n=== Classification Report XGBoost (Tuned) ===")
print(classification_report(y_test, y_pred_xgb_tuned))


=== Confusion Matrix XGBoost (Tuned) ===
[[48429 33688]
 [ 3583  7467]]

=== Classification Report XGBoost (Tuned) ===
              precision    recall  f1-score   support

           0       0.93      0.59      0.72     82117
           1       0.18      0.68      0.29     11050

    accuracy                           0.60     93167
   macro avg       0.56      0.63      0.50     93167
weighted avg       0.84      0.60      0.67     93167



In [ ]:
import joblib

joblib.dump(model_xgb_tuned, '../model/model_xgb_credit_risk.pkl')

joblib.dump(scaler, '../model/scaler_credit_risk.pkl')

print("Model dan Scaler berhasil disimpan!")

Model dan Scaler berhasil disimpan!


In [30]:
print(X_train.columns.tolist())

['loan_amnt', 'term', 'int_rate', 'sub_grade', 'emp_length', 'annual_inc', 'verification_status', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'mths_since_earliest_cr_line', 'home_ownership_MORTGAGE', 'home_ownership_NONE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_educational', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_renewable_energy', 'purpose_small_business', 'purpose_vacation', 'purpose_wedding']
